In [ ]:
PROJECT_REF = "main"


def _install_project(project_ref: str):
    import urllib.parse
    import urllib.request

    from google.colab import userdata

    github_token = userdata.get("GITHUB_TOKEN_JLENS_REAS")
    if not github_token:
        raise RuntimeError(
            "Required Colab secret GITHUB_TOKEN_JLENS_REAS is unavailable"
        )

    query = urllib.parse.urlencode({"ref": project_ref})
    bootstrap_url = (
        "https://api.github.com/repos/noamdwc/jlens-reasoning/"
        "contents/scripts/colab_bootstrap.py?" + query
    )
    request = urllib.request.Request(
        bootstrap_url,
        headers={
            "Authorization": f"Bearer {github_token}",
            "Accept": "application/vnd.github.raw+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    try:
        with urllib.request.urlopen(request) as response:
            bootstrap_source = response.read().decode("utf-8")
    except Exception:
        raise RuntimeError("Unable to load the Colab bootstrap") from None

    namespace = {}
    exec(
        compile(
            bootstrap_source,
            "scripts/colab_bootstrap.py",
            "exec",
        ),
        namespace,
    )
    return namespace["bootstrap"](
        project_ref=project_ref,
        github_token=github_token,
    )


PROJECT_DIR = _install_project(PROJECT_REF)
del _install_project

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(require_cuda=True)
context

In [ ]:
import subprocess

commit = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

probe = context.runs_dir / "environment-check.txt"
probe.write_text("colab environment check\n", encoding="utf-8")

print(f"Commit: {commit}")
print(f"Device: {context.device}")
print(f"Artifact root: {context.artifact_root}")
print(f"Drive write probe: {probe}")
print(f"W&B enabled: {context.wandb_enabled}")